# Basics of LLM Fine-Tuning (Beginner-Friendly, Colab Ready)

This notebook mirrors the lecture in `docs/chapter_llm_ft/basics.md` and shows a minimal, reliable workflow to fine-tune an open LLM using the Hugging Face ecosystem. We include small, runnable demos that work on Google Colab.

What you will learn:
- Install the right libraries and pick a manageable model for Colab
- Prepare a tiny instruction dataset in the OpenAI "messages" format
- Load the model with 4-bit quantization and optionally add LoRA adapters
- Run supervised fine-tuning with `trl.SFTTrainer`
- Do quick inference to verify the result

If your GPU VRAM is limited, we will default to a very small chat model (TinyLlama) so you can complete the tutorial on free Colab. If you have an A100 or >24GB VRAM, you can switch to Llama 3 8B as in the lecture.


In [ ]:
# Install core libraries (quietly) and print versions
import sys, subprocess, pkgutil

def pip_install(packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *packages])

pip_install([
    "transformers",
    "datasets",
    "accelerate",
    "trl",
    "peft",
    "bitsandbytes",
    "tensorboard",
])

import torch, transformers, datasets, accelerate, trl, peft
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)


## Check your runtime

- If you are on Colab, go to Runtime → Change runtime type → Select GPU.
- TinyLlama runs on free T4. Larger models (e.g., Llama 3 8B) need >= 20–24GB VRAM.

Run the next cell to see your GPU info.


In [ ]:
import torch, os
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())


## Choose a model (small default for Colab)

For free Colab, we use a tiny chat model so training runs quickly. If you have more VRAM, switch to Llama 3 8B as in the lecture. We also set up 4-bit quantization to reduce memory needs.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Use a very small, public model for Colab. Swap to Llama 3 8B if you have VRAM.
small_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# For higher VRAM setups:
# big_model_id = "meta-llama/Meta-Llama-3-8B"

# Pick compute dtype based on GPU capability (Colab T4 → float16)
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

model_id = small_model_id

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
)
# Ensure config dtype is set consistently for generation
try:
    model.config.torch_dtype = compute_dtype
except Exception:
    pass


## Prepare a tiny instruction dataset (messages format)

We use a minimal OpenAI-style `messages` structure:
- `system`: high-level behavior
- `user`: prompt ending with "Answer:"
- `assistant`: final answer only (keep units/precision consistent)

This makes tokenization and evaluation easier later.


In [ ]:
import json

demo_records = [
    {
        "messages": [
            {"role": "system", "content": "You are a clinical calculator assistant."},
            {"role": "user", "content": "Patient Note: BMI example.\nQuestion: Height 1.75m, Weight 70kg.\nAnswer:"},
            {"role": "assistant", "content": "22.86"}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "You are a clinical calculator assistant."},
            {"role": "user", "content": "Patient Note: adolescent with hypertension.\nQuestion: Compute creatinine clearance (Cockcroft-Gault).\nAnswer:"},
            {"role": "assistant", "content": "95"}
        ]
    },
]

with open("train_dataset.json", "w") as f:
    for r in demo_records:
        f.write(json.dumps(r) + "\n")

!head -n 2 train_dataset.json


## Optional: Add LoRA (PEFT)

We add small trainable adapters to reduce memory and speed up training. You can skip this if you want to fine-tune all parameters (not recommended on Colab).


In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
# Align adapter dtype with desired compute dtype (float16 on T4)
try:
    model = model.to(compute_dtype)
except Exception:
    pass
model.print_trainable_parameters()


## Supervised Fine-Tuning with TRL SFTTrainer

We use `SFTTrainer` because it:
- Supports `messages` format directly
- Handles prompt masking and PEFT
- Provides a simple training interface

We start with tiny settings so this runs in a few minutes on Colab.


In [ ]:
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer, setup_chat_format

# Ensure chat formatting only if missing; also ensure pad token
if getattr(tokenizer, "chat_template", None) is None:
    model, tokenizer = setup_chat_format(model, tokenizer)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

# Load our tiny JSONL dataset
train_ds = load_dataset("json", data_files="train_dataset.json", split="train")

# Set precision flags compatible with Colab T4
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
args = TrainingArguments(
    output_dir="tinyllama-basics-sft",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    bf16=use_bf16,
    fp16=not use_bf16,
    tf32=False,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
)

import inspect

sft_kwargs = dict(
    model=model,
    train_dataset=train_ds,
    args=args,
    max_seq_length=1024,
    packing=True,
    dataset_kwargs={"add_special_tokens": False, "append_concat_token": False},
)
params = inspect.signature(SFTTrainer.__init__).parameters
if "tokenizer" in params:
    sft_kwargs["tokenizer"] = tokenizer
elif "processing_class" in params:
    sft_kwargs["processing_class"] = tokenizer

trainer = SFTTrainer(**sft_kwargs)

trainer.train()
trainer.save_model()


## Prepare model for inference (disable checkpointing)

During training we enable gradient checkpointing to save memory. For inference, we disable it and enable caching to avoid warnings and speed up generation.


## Quick inference

We ask a question that matches our template and extract the model's response. Keep expectations modest: with 1 epoch on a tiny dataset, this is just a pipeline sanity check.


In [ ]:
import re

# Switch to eval mode and disable grad checkpointing + enable cache for gen
model.eval()
try:
    model.gradient_checkpointing_disable()
except Exception:
    pass
if hasattr(model.config, "use_cache"):
    model.config.use_cache = True
# Enforce float16 on T4 for safe generation
try:
    model = model.to(compute_dtype)
    model.config.torch_dtype = compute_dtype
except Exception:
    pass


def generate(prompt, max_new_tokens=64):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

user_q = (
    "Patient Note: BMI example.\n"
    "Question: Height 1.75m, Weight 70kg.\n"
    "Answer:"
)
full_prompt = f"You are a clinical calculator assistant.\n\n{user_q}"
text = generate(full_prompt)
print(text)
